In [1]:
%run parse_data.ipynb

In [2]:
from IPython.display import display,Markdown #,HTML
import numpy as np
from scipy import stats
from IPython.display import display,Markdown #,HTML
from matplotlib import pyplot as plt
import pandas as pd
import pingouin as pg

In [3]:
def display_title(s, pref='Figure', num=1, center=False):
    ctag = 'center' if center else 'p'
    s    = f'<{ctag}><span style="font-size: 1.2em;"><b>{pref} {num}</b>: {s}</span></{ctag}>'
    if pref=='Figure':
        s = f'<br>{s}'
    else:
        s = f'<br>{s}'
    display( Markdown(s) )

def corrcoeff(x, y):
    r = np.corrcoef(x, y)[0,1]
    return r

def plot_regression_line(ax, x, y, **kwargs):
    a,b   = np.polyfit(x, y, deg=1)
    x0,x1 = min(x), max(x)
    y0,y1 = a*x0 + b, a*x1 + b
    ax.plot([x0,x1], [y0,y1], **kwargs)

From the Variance Explained score of our descriptive PCA, we can see that Promotion have the highest ratio. The ratio shows that Promotion is highly varied across the dataset.

High variation usually calls for customer segmentation in business. We will intend use One-way ANOVA to test the hypothesis if segmentation in Promotion will increase the score of IBB, which in turn leads to higher BNPL usage rate 

First, we will divide the dataset into 3 groups: low, medium and high promotion score

In [4]:
# Using pd.cut to decide the width of the interval
promotion = df[['P', 'IBB']]

pd.cut(promotion.values.ravel(), 3)

[(2.333, 3.667], (2.333, 3.667], (2.333, 3.667], (0.996, 2.333], (2.333, 3.667], ..., (0.996, 2.333], (3.667, 5.0], (0.996, 2.333], (0.996, 2.333], (2.333, 3.667]]
Length: 1606
Categories (3, interval[float64, right]): [(0.996, 2.333] < (2.333, 3.667] < (3.667, 5.0]]

In [5]:
# Divide into groups
low_promotion = promotion.loc[promotion['P'] <= 2.333]
med_promotion = promotion.loc[(2.333 < promotion['P']) & (promotion['P'] <= 3.667)]
high_promotion = promotion.loc[promotion['P'] > 3.667]

Since the number of events in each interval is unequal, with over 1/2 of respondents falls into low promotion category, we will use Levene's test to know if 3 groups have homogeneous variance to be eligible for One-way ANOVA test

In [6]:
stats.levene(low_promotion['IBB'], med_promotion['IBB'], high_promotion['IBB'])

LeveneResult(statistic=4.779478575762907, pvalue=0.008641753676887572)

p value = 0.008 < 0.05 

We reject H0 and there is inhomogeneous variance between 3 groups. In this case, we will switch to Welch's ANOVA test for the groups.

In [7]:
bin = {1: 'Low', 2: 'Medium', 3: 'High'}

def p_segment(v):
    if v <= 2.333:
        return bin[1] 
    elif 2.333 < v <= 3.667:
        return bin[2] 
    else:
        return bin[3]

df['P_Segment'] = df['P'].apply(p_segment)


a = pg.welch_anova(data = df, dv = 'IBB', between = 'P_Segment')
welch_anova_result = pd.DataFrame(a)
welch_anova_result = welch_anova_result.round(4)
display(welch_anova_result)

,Source,ddof1,ddof2,F,p-unc,np2
0,P_Segment,2,353.2252,50.7753,0.0,0.1047


From the results, extreme values of F and p value reflects the nature of the data collected

p value  $\alpha$ < 0.05 -> We reject the H0 hypothesis, that is the mean score for IBB is the same throughout all categories. The test indicates our hypothesis if segmentation in Promotion will increase the IBB score is correct.

Further test to confirm where the main difference comes from includes Games-Howell Post Hoc tests

In [8]:
def difference_table():
    global welch_anova_result
    display_title("Welch's ANOVA test for IBB mean score throughout Promotion categories (Low, Medium, High)", pref='Table', num=1, center=False)
    display(welch_anova_result)

    display_title('Games-Howell Post Hoc Results for segmented Promotion respondents', pref='Table', num=2, center=False)
    post_hoc = pg.pairwise_gameshowell(data = df, dv = 'IBB', between = 'P_Segment')
    post_hoc = post_hoc.rename(columns={'diff': 'Difference'})
    post_hoc = post_hoc.round(2)
    display(post_hoc)

difference_table()

<br><p><span style="font-size: 1.2em;"><b>Table 1</b>: Welch's ANOVA test for IBB mean score throughout Promotion categories (Low, Medium, High)</span></p>

,Source,ddof1,ddof2,F,p-unc,np2
0,P_Segment,2,353.2252,50.7753,0.0,0.1047


<br><p><span style="font-size: 1.2em;"><b>Table 2</b>: Games-Howell Post Hoc Results for segmented Promotion respondents</span></p>

,A,B,mean(A),mean(B),Difference,se,T,df,pval,hedges
0,High,Low,3.26,2.53,0.73,0.07,9.89,281.90,0.0,0.87
1,High,Medium,3.26,2.88,0.38,0.09,4.44,329.18,0.0,0.48
2,Low,Medium,2.53,2.88,-0.35,0.07,-4.90,371.29,0.0,-0.41


We can see that the main difference comes from the High and Low Promotion respondents, which is totally expected. However, the second highest difference comes from the High and Medium respondents; this information is crucial to BNPL services in further segmenting their customer base or potential customers. 

---

The correlation graph between independent variables and IBB were drafted in the descriptive section, yet in statistics, correlation does not always equal to causation. 

We will use hypothesis testing to test if decreasing or increasing the the IVs score will have an actual effect on IBB score. The test we use will OLS model, including the control variables to validate 

In [9]:
from statsmodels.api import add_constant
from statsmodels.regression import linear_model

X_nocontrol = df[['P','SI','H','SC','NE']]
X_control = add_constant(X_nocontrol)

X_control = df[['P','SI','H','SC','NE','Income','Gender','Status']]
X_control = add_constant(X_control)

y = df['IBB']

model0 = linear_model.OLS(y, X_nocontrol).fit()
model1 = linear_model.OLS(y, X_control).fit()

In [10]:
def ols_hypothesis():
    display_title("Hypothesis testing for associations with Impulsive Buying Behaviour - no demographic variables", pref='Table', num=3, center=False)
    a = model0.summary().tables[1]
    print(a)
    
    display_title("Hypothesis testing for associations with Impulsive Buying Behaviour - with demographic variables", pref='Table', num=4, center=False)
    b = model1.summary().tables[1]
    print(b)

ols_hypothesis()

<br><p><span style="font-size: 1.2em;"><b>Table 3</b>: Hypothesis testing for associations with Impulsive Buying Behaviour - no demographic variables</span></p>

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
P              0.0417      0.038      1.100      0.272      -0.033       0.116
SI             0.3790      0.042      8.990      0.000       0.296       0.462
H              0.4103      0.029     14.220      0.000       0.354       0.467
SC            -0.0167      0.027     -0.613      0.540      -0.070       0.037
NE             0.0990      0.028      3.582      0.000       0.045       0.153


<br><p><span style="font-size: 1.2em;"><b>Table 4</b>: Hypothesis testing for associations with Impulsive Buying Behaviour - with demographic variables</span></p>

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.6314      0.217     12.150      0.000       2.206       3.056
P              0.0355      0.036      0.976      0.329      -0.036       0.107
SI             0.2341      0.041      5.775      0.000       0.155       0.314
H              0.1662      0.033      4.977      0.000       0.101       0.232
SC            -0.2828      0.033     -8.527      0.000      -0.348      -0.218
NE            -0.0856      0.029     -2.925      0.004      -0.143      -0.028
Income         0.0014      0.015      0.095      0.925      -0.028       0.030
Gender         0.0369      0.057      0.643      0.520      -0.076       0.150
Status         0.0148      0.069      0.216      0.829      -0.120       0.150


Social Influence and Happiness is statistically significant and we can imply **positive** causation effects on Impulsive Buying Behavior.

Self-Control and Normative Evaluation is statistically significant and we can imply **negative** causation effects on Impulsive Buying Behaviour respectively. 

Promotion is not statistically significant after adding demographic variables into the model, so we cannot make any conclusion on its effects on Impulsive Buying Behaviour

Income, Gender, and Status are statistically significant in predicting IBB score, making them valid control variables.

---

There have been multiple studies in the past shows that people tend to overestimate their ability to control impulse [(Nordgren, 2007)](https://homepages.se.edu/cvonbergen/files/2013/01/The-Restraint-Bias_How-the-Illusion-of-Self-Restraint-Promotes-Impulsive-Behavior.pdf). This topic is highly relevent to BNPL providers since BNPL services usually thrives when people are highly impulsive. However, providers also need to balance the impulse of their customers to avoid customer burnout and negative impression from non-users. 

We will use two-sample t test to test the hypothesis if respondents with low self-control score overestimate their impulse control more than respondents with high self-control score. 

In [11]:
# define the bin value for low self-control and high self-control respondents
self_control = df[['SC', 'NE']]

pd.cut(promotion.values.ravel(), 2)

[(0.996, 3.0], (0.996, 3.0], (3.0, 5.0], (0.996, 3.0], (0.996, 3.0], ..., (0.996, 3.0], (3.0, 5.0], (0.996, 3.0], (0.996, 3.0], (0.996, 3.0]]
Length: 1606
Categories (2, interval[float64, right]): [(0.996, 3.0] < (3.0, 5.0]]

In [12]:
# divide into groups
low_sc = self_control.loc[self_control['SC'] <= 3]
high_sc = self_control.loc[(3 < self_control['SC']) & (self_control['SC'] <= 5)]

We once again use Levene's test to test for homogenity in variance between the 2 groups

In [13]:
print(stats.levene(low_sc['NE'], high_sc['NE']))

LeveneResult(statistic=6.607339766163802, pvalue=0.010335514646528009)


P-value  $\alpha$ < 0.05, so we reject H0. There is inhomogenity in variance between the groups. 

We will use a variation of standard t-test which is Welch's t-test to test our hypothesis in this case.

In [14]:
NE_ttest = stats.ttest_ind(high_sc['NE'], low_sc['NE'], equal_var = False)

def NE_result(num = 3):
    NE_visual = pd.DataFrame()
    NE_visual['t-statistic'] = [NE_ttest.statistic]
    NE_visual['p-value'] = [NE_ttest.pvalue]
    display_title("Welch's t-test result for Normative Evaluation between high and low Self-Control groups: T-statistics and P-value", pref='Table', num=num, center=False)
    NE_visual = NE_visual.round(2)
    return NE_visual

NE_result(num = 3)

<br><p><span style="font-size: 1.2em;"><b>Table 3</b>: Welch's t-test result for Normative Evaluation between high and low Self-Control groups: T-statistics and P-value</span></p>

,t-statistic,p-value
0,2.46,0.01


P-value  $\alpha$ < 0.05, so we reject H0. The difference between the mean for Normative Evaluation score between both groups is statistically significant.

t-statistic value shows the direction of the difference. t-statistic = 2.464 > 0, so we can conclude that the Normative Evaluation mean score in high Self-Control groups is higher than that in low Self-Control groups.

This rejects the hypothesis of low self-control score overestimate their impulse control more than respondents with high self-control score among the respondents.